# 21. 학습 데이터 세트 스캔 (Training Dataset Scan)

이 노트북은 모델 학습 직전, 이전 단계(18, 19, 20번)에서 생성된 데이터들이 의도대로 준비되었는지 최종 점검합니다.

**점검 대상:**
1. **가공 완료된 원본 데이터**: `train`, `val_tune`, `val_calib`, `test`
2. **학습용 서브셋 (19번)**: 언더배깅 분할 데이터 (`subset_0~9`)
3. **샘플링 검증셋 (20번)**: 1:100 가중치 샘플링 데이터

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
from pathlib import Path
import config.train_config as cfg
import config.eval_config as ecfg

def get_stats(df, name):
    pos = df[cfg.TARGET_COL].sum()
    neg = len(df) - pos
    return {
        'Dataset': name,
        'Total Rows': f"{len(df):,}",
        'Positive (Failure)': f"{pos:,}",
        'Negative (Normal)': f"{neg:,}",
        'Ratio (1:N)': f"1:{neg/pos:.1f}" if pos > 0 else "N/A"
    }

print("✅ 환경 준비 완료")

## 1. 원본 가공 데이터 점검
Feature Engineering이 완료된 전체 데이터셋 현황입니다.

In [ ]:
paths_raw = [
    ('Train (Full)', cfg.TRAIN_PATH),
    ('Val Tune (Full)', cfg.VAL_TUNE_PATH),
    ('Val Calib (Full)', ecfg.VAL_CALIB_PATH),
    ('Test (Full)', ecfg.TEST_PATH)
]

stats_list = []
for name, p in paths_raw:
    if Path(p).exists():
        df = pd.read_parquet(p, columns=[cfg.TARGET_COL])
        stats_list.append(get_stats(df, name))
    else:
        print(f"❌ [Missing] {name}: {p}")

pd.DataFrame(stats_list)

## 2. 학습용 서브셋 점검 (19번 결과)
10:1 비율로 언더샘플링된 학습용 서브셋입니다.

In [ ]:
subset_path = Path(cfg.SUBSET_DIR)
subset_files = sorted(list(subset_path.glob("subset_*.parquet")))

if not subset_files:
    print(f"❌ [Missing] {cfg.SUBSET_DIR} 에 서브셋 파일이 없습니다.")
else:
    sub_stats = []
    for f in [subset_files[0], subset_files[-1]]:
        df = pd.read_parquet(f, columns=[cfg.TARGET_COL])
        sub_stats.append(get_stats(df, f.name))
    
    print(f"✅ 총 {len(subset_files)}개의 서브셋 감지됨")
    display(pd.DataFrame(sub_stats))

## 3. 샘플링 검증셋 점검 (20번 결과)
원본 Val Tune에서 전략적으로 추출된 고밀도 검증셋입니다.

In [ ]:
val_sampled_path = Path(cfg.VAL_TUNE_SAMPLED_PATH)

if val_sampled_path.exists():
    df_val = pd.read_parquet(val_sampled_path)
    print(f"✅ 샘플링 검증셋 로드 완료: {val_sampled_path.name}")
    
    # 원본 Val Tune과 비교
    full_val_df = pd.read_parquet(cfg.VAL_TUNE_PATH, columns=[cfg.TARGET_COL])
    
    comparison = [
        get_stats(full_val_df, 'Val Tune (Full)'),
        get_stats(df_val, 'Val Tune (Sampled)')
    ]
    
    print("\n📊 원본 vs 샘플링 비교:")
    display(pd.DataFrame(comparison))
    
    print("\n🔍 샘플링 데이터 요약:")
    print(f"  - 총 행 수: {len(df_val):,}")
    print(f"  - 고장 개수: {df_val[cfg.TARGET_COL].sum():,}")
    display(df_val.head())
else:
    print(f"❌ [Missing] 샘플링 검증셋이 없습니다: {val_sampled_path}")

## 4. 최종 결론
모든 데이터셋의 'Positive' 개수가 원본과 일치하는지 확인하세요. 
특히 **원본 Val Tune**과 **샘플링 검증셋**의 고장 데이터 개수가 동일해야 정상입니다.